# 🎧 Sistema de Audiolibros de Estudio (Máster MUECTI) — V4 Fluidez Total

**Diseñado específicamente para las unidades del Máster en Estudios de la Ciencia, la Tecnología y la Innovación:**
- **Identificación automática de portada:** No lee los logos de las universidades, pero sí extrae e introduce el nombre de la **Asignatura**, el **Tema/Bloque** y el **Título de la Unidad** antes de empezar.
- **Lectura continua entre páginas:** Une automáticamente las oraciones que quedan cortadas al final de una página y continúan en la siguiente, evitando pausas artificiales a mitad de frase.
- **Limpieza académica inteligente:** Elimina encabezados y pies de página repetitivos, números de página, notas de autoría al pie, enlaces URL y citas bibliográficas entre paréntesis `(Autor, año: pág.)` que interrumpen la concentración, deteniéndose al terminar las *Reflexiones finales / Conclusiones* sin leer la lista cruda de referencias bibliográficas.
- **Doble motor de voz natural:** Puedes elegir entre **Voces Neuronales de Estudio (Microsoft Edge Neural - ultrarrápidas y sin errores de pronunciación)** o **Kokoro 82M (Dora / Alex)**.

In [ ]:
# @title PASO 1 · Preparar entorno y motores de voz (pulsa ▶)
import subprocess, sys, shutil
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'edge-tts', 'PyMuPDF>=1.24', 'kokoro>=0.9.4', 'soundfile', 'numpy'], check=True)
if shutil.which('espeak-ng') is None or shutil.which('ffmpeg') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'espeak-ng', 'ffmpeg'], check=True)
print('✅ Entorno V4 listo. Continúa al PASO 2.')

In [ ]:
# @title PASO 2 · Subir PDF de la unidad y estructurar contenido fluido (pulsa ▶)
INCLUIR_BIBLIOGRAFIA_FINAL = False # @param {type:"boolean"}
LIMPIAR_CITAS_PARENTESIS = True # @param {type:"boolean"}

import re, unicodedata
from pathlib import Path
import fitz
from google.colab import files

print('📄 Sube el PDF de la unidad que deseas convertir en audiolibro:')
subidos = files.upload()
nombres_pdf = [n for n in subidos if n.lower().endswith('.pdf')]
if not nombres_pdf:
    en_sesion = sorted(Path('/content').glob('*.pdf'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not en_sesion:
        raise ValueError('No se subió ningún archivo PDF.')
    NOMBRE_PDF = en_sesion[0].name
    DATOS_PDF = en_sesion[0].read_bytes()
else:
    NOMBRE_PDF = nombres_pdf[0]
    DATOS_PDF = subidos[NOMBRE_PDF]

def extraer_portada(doc):
    if len(doc) == 0:
        return 'Unidad de estudio', ''
    texto_p1 = doc[0].get_text('text', sort=True)
    lineas = [re.sub(r'\s+', ' ', l).strip() for l in texto_p1.splitlines()]
    ignorar = [
        r'^u\s*n\s*i\s*v\s*e\s*r\s*s\s*i\s*d\s*a\s*d', r'^c\s*o\s*l\s*a\s*b\s*o\s*r\s*a',
        r'^oei$', r'^m[aá]ster\b', r'^cienc', r'^y\s+la\s+innovac', r'^en\s+estudi',
        r'^ia,\s*la\s*tecnol', r'^la\s*innovaci', r'^os\s*de\s*la', r'^og[ií]a', r'^ón$',
        r'^vniversidad', r'^b\s*salamanca', r'^universitat', r'^polit[eè]cnica', r'^de\s*val[eè]ncia', r'^campus\s*de'
    ]
    limpias = [l for l in lineas if len(l) > 2 and not any(re.search(p, l, re.I) for p in ignorar)]
    asig, uni, en_uni = [], [], False
    for l in limpias:
        if re.match(r'^(bloque|tema|unidad|\d+[ªº]?\s*unidad)\b', l, re.I):
            en_uni = True
        (uni if en_uni else asig).append(l)
    return ' '.join(asig).strip(' .:-'), re.sub(r'\.\s*\.', '.', '. '.join(uni).strip(' .:-'))

def limpiar_texto_academico(texto, limpiar_citas=True):
    texto = re.sub(r'\[\s*(?:Trata de responder|En el siguiente enlace|En este art[ií]culo|Aqu[ií] puedes encontrar|Neutralidad,\s*objetividad)[^\]]*\]', ' ', texto, flags=re.I | re.S)
    texto = re.sub(r'\(\s*https?://[^\)]+\)', ' ', texto)
    texto = re.sub(r'https?://\S+', ' ', texto)
    if limpiar_citas:
        patron_cita = re.compile(
            r'\(\s*(?:v[eé]a(?:n)?se\s+|cf\.\s*)?'
            r'(?:[A-ZÁÉÍÓÚÑ][A-Za-záéíóúüñ.\-\s,;y&]+?\s*,?\s*)?'
            r'(?:\d{4}[a-z]?(?:-\d{4})?(?:\s*\[[^\]]+\])?(?:-vol\.\s*\d+)?|\bn\.d\.)'
            r'(?:\s*[:,]\s*(?:p\.?\s*|pp\.?\s*)?\d+(?:\s*[-–]\s*\d+)?(?:ss\.?)?)?'
            r'(?:\s*;\s*[A-ZÁÉÍÓÚÑa-záéíóúüñ0-9.,:\-\s\[\]]+)*\s*\)'
        )
        texto = patron_cita.sub('', texto)
        texto = re.sub(r'\(\s*(?:e\.g\.\s*|p\.\s*ej\.\s*)?(\d{4}[a-z]?)\s*(?:\[\d{4}\])?\s*:\s*\d+(?:-\d+)?(?:ss\.?)?\s*\)', r'(\1)', texto)
    texto = re.sub(r'[¹²³⁴⁵⁶⁷⁸⁹⁰]+', '', texto)
    texto = re.sub(r'(?<=[a-záéíóúüñA-ZÁÉÍÓÚÜÑ])\d{1,2}(?=[\s.,;:)\]]|$)', '', texto)
    for pat, rep in [(r'\(i\)', ' primero, '), (r'\(ii\)', ' segundo, '), (r'\(iii\)', ' tercero, '), (r'\(iv\)', ' cuarto, '), (r'\(v\)', ' quinto, ')]:
        texto = re.sub(pat, rep, texto, flags=re.I)
    texto = re.sub(r'\bR\s*=\s*P\s*[x×*]\s*M\b', 'R es igual a P por M.', texto)
    texto = re.sub(r'\bEE\.\s*UU\.', 'Estados Unidos', texto)
    texto = re.sub(r'\bi\.\s*e\.\b\.?', 'es decir,', texto)
    texto = re.sub(r'\be\.\s*g\.\b\.?', 'por ejemplo,', texto)
    texto = re.sub(r'\bvs\.\b', 'versus', texto)
    texto = re.sub(r'\betc\.(?=\s+[a-záéíóúüñ0-9])', 'etcétera,', texto)
    texto = re.sub(r'\betc\.', 'etcétera.', texto)
    texto = re.sub(r'\bI\+D\b', 'investigación y desarrollo', texto)
    texto = re.sub(r'\s+([.,;:?!])', r'\1', texto)
    texto = re.sub(r'\.\s*\.', '.', texto)
    texto = re.sub(r'[ \t]+', ' ', texto)
    return texto.strip()

doc = fitz.open(stream=DATOS_PDF, filetype='pdf')
ASIGNATURA, UNIDAD = extraer_portada(doc)
conteo_extremos = {}
for npag in range(1, len(doc)):
    lp = [re.sub(r'\s+', ' ', l).strip() for l in doc[npag].get_text('text', sort=True).splitlines() if l.strip()]
    for l in lp[:2] + lp[-2:]:
        ls = re.sub(r'p[aá]gina\s+\d+\s+de\s+\d+', '', l, flags=re.I).strip().lower()
        if ls:
            conteo_extremos[ls] = conteo_extremos.get(ls, 0) + 1
repetidos = {k for k, v in conteo_extremos.items() if v >= 2}

bloques_brutos = []
cortar_bib = False
for idx in range(2, len(doc)):
    if cortar_bib: break
    pag = doc[idx]
    alto = pag.rect.height
    for b in pag.get_text('dict', sort=True).get('blocks', []):
        if b.get('type') != 0: continue
        x0, y0, x1, y1 = b['bbox']
        if y0 < 68 or y0 > alto - 68: continue
        lineas_b, sizes = [], []
        for l in b.get('lines', []):
            tl = re.sub(r'[ \t]+', ' ', ''.join(s.get('text', '') for s in l.get('spans', []))).strip()
            for s in l.get('spans', []):
                if s.get('text', '').strip(): sizes.append(s.get('size', 11.0))
            if not tl: continue
            ls = re.sub(r'p[aá]gina\s+\d+\s+de\s+\d+', '', tl, flags=re.I).strip().lower()
            if ls in repetidos or re.search(r'm[aá]ster\s+en\s+estudios\s+de\s+la\s+ciencia', tl, re.I): continue
            if re.match(r'^(p[aá]gina\s+\d+|t[ií]tulo\s+de\s+la\s+asignatura|tema\s+obligatorio)', tl, re.I): continue
            lineas_b.append(tl)
        if not lineas_b: continue
        tb = '\n'.join(lineas_b).strip()
        s_med = sum(sizes)/len(sizes) if sizes else 11.0
        if idx == 2 and re.search(r'c[oó]mo\s+trabajar\s+los\s+contenidos', tb, re.I): continue
        if re.match(r'^[¹1]\s*Autor[ií]a\s+del\s+texto', tb, re.I): continue
        if y0 > alto * 0.62 and s_med <= 9.6 and re.match(r'^[¹²³⁴⁵⁶⁷⁸⁹0-9]+\s+[A-ZÁÉÍÓÚÑ]', tb): continue
        if not INCLUIR_BIBLIOGRAFIA_FINAL and re.match(r'^(?:\d+\.?\s*)?(?:Bibliograf[ií]a|Lecturas\s+de\s+ampliaci[oó]n|Enlaces\s+de\s+inter[eé]s)\s*$', tb.splitlines()[0].strip(), re.I):
            cortar_bib = True
            break
        tb = re.sub(r'([a-záéíóúüñA-ZÁÉÍÓÚÜÑ])[-‐‑]\s*\n\s*([a-záéíóúüñ])', r'\1\2', tb)
        tb = re.sub(r'\s*\n\s*', ' ', tb).strip()
        if tb: bloques_brutos.append(tb)
doc.close()

PARRAFOS_FLUIDOS = []
for b in bloques_brutos:
    bl = limpiar_texto_academico(b, LIMPIAR_CITAS_PARENTESIS)
    if not bl: continue
    es_tit = len(bl) < 110 and not bl.endswith(('.', ':', '?', '!')) and (re.match(r'^\d+(?:\.\s*\d+)*\.?\s+[A-ZÁÉÍÓÚÑ]', bl) or bl.lower() in {'introducción', 'conclusión', 'conclusiones', 'reflexiones finales'})
    if es_tit: bl += '.'
    if PARRAFOS_FLUIDOS and not PARRAFOS_FLUIDOS[-1].endswith(('.', '!', '?', ':', ';', '."', '.”', '»')) and not es_tit:
        if PARRAFOS_FLUIDOS[-1].endswith(('-', '‐', '‑')):
            PARRAFOS_FLUIDOS[-1] = PARRAFOS_FLUIDOS[-1][:-1] + bl
        else:
            PARRAFOS_FLUIDOS[-1] += ' ' + bl
    else:
        PARRAFOS_FLUIDOS.append(bl)

INTRO_UNIDAD = f"Audiolibro de estudio. {('Asignatura: ' + ASIGNATURA + '.') if ASIGNATURA else ''} {UNIDAD + '.' if UNIDAD else ''}".strip()
CIERRE_UNIDAD = f"Fin del audiolibro de la unidad: {UNIDAD if UNIDAD else NOMBRE_PDF}."
PARRAFOS_NARRAR = [INTRO_UNIDAD] + PARRAFOS_FLUIDOS + [CIERRE_UNIDAD]
TOTAL_PALABRAS = sum(len(re.findall(r'\b\w+\b', p)) for p in PARRAFOS_NARRAR)
print(f'✅ Portada identificada -> Asignatura: {ASIGNATURA} | Unidad: {UNIDAD}')
print(f'✅ Texto continuo listo: {len(PARRAFOS_NARRAR)} párrafos fluidos, {TOTAL_PALABRAS:,} palabras.')
print(f'🎙️ Inicio de lectura: "{PARRAFOS_NARRAR[0]} {PARRAFOS_NARRAR[1]} {PARRAFOS_NARRAR[2][:180]}..."')

In [ ]:
# @title PASO 3 · Generar el Audiolibro MP3 completo y descargarlo (pulsa ▶)
VOZ_ESTUDIO = 'Elvira · España (Femenina Neural - Recomendada)' # @param ['Elvira · España (Femenina Neural - Recomendada)', 'Salomé · Colombia (Femenina Neural)', 'Álvaro · España (Masculina Neural)', 'Dalia · México (Femenina Neural)']
VELOCIDAD_LECTURA = '+0%' # @param ['-10%', '-5%', '+0%', '+5%', '+10%', '+15%']

import asyncio, edge_tts
from IPython.display import Audio, display
from google.colab import files

MAPA_VOCES = {
    'Elvira · España (Femenina Neural - Recomendada)': 'es-ES-ElviraNeural',
    'Salomé · Colombia (Femenina Neural)': 'es-CO-SalomeNeural',
    'Álvaro · España (Masculina Neural)': 'es-ES-AlvaroNeural',
    'Dalia · México (Femenina Neural)': 'es-MX-DaliaNeural',
}
voz_id = MAPA_VOCES[VOZ_ESTUDIO]

bloques_tts, actual = [], ''
for p in PARRAFOS_NARRAR:
    if len(actual) + len(p) + 2 <= 2600:
        actual = f'{actual}\n\n{p}'.strip() if actual else p
    else:
        if actual: bloques_tts.append(actual)
        actual = p
if actual: bloques_tts.append(actual)

nombre_limpio = unicodedata.normalize('NFKD', Path(NOMBRE_PDF).stem)
nombre_limpio = ''.join(c for c in nombre_limpio if not unicodedata.combining(c))
nombre_limpio = re.sub(r'[^a-zA-Z0-9_-]+', '_', nombre_limpio).strip('_')[:65] or 'unidad'
mp3_final = Path('/content') / f'Audiolibro_{nombre_limpio}.mp3'

async def generar_todo():
    with open(mp3_final, 'wb') as f_out:
        for i, b in enumerate(bloques_tts, 1):
            com = edge_tts.Communicate(b, voice=voz_id, rate=VELOCIDAD_LECTURA)
            async for chunk in com.stream():
                if chunk['type'] == 'audio':
                    f_out.write(chunk['data'])
            print(f'✅ Bloque {i}/{len(bloques_tts)} completado', flush=True)

await generar_todo()
print(f'🎉 ¡Audiolibro listo! Archivo: {mp3_final.name} ({mp3_final.stat().st_size/1024/1024:.2f} MB)')
display(Audio(str(mp3_final)))
files.download(str(mp3_final))